In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(".")
print(" Ready")

 Ready


In [2]:
%%writefile agents/web_search_agent.py
import time
from typing import Dict, List

from tools.llm_clients import research_with_fallback
from tools.logger import log_agent_event

AGENT_NAME = "web_search_agent"
MAX_QUERIES = 5


def _dedupe_sources(sources: List[Dict]) -> List[Dict]:
    seen, unique = set(), []
    for s in sources:
        url = s.get("url")
        if url and url not in seen:
            seen.add(url)
            unique.append(s)
    return unique


def _is_rate_limit_error(e: Exception) -> bool:
    """Detects rate-limit errors across providers by message content, since
    exception TYPES differ between Groq/Gemini/Tavily SDKs but the signal
    (429, 'rate limit', 'RESOURCE_EXHAUSTED') is consistent enough to check
    as text. Found necessary after Day 15's benchmark showed parallel calls
    trigger this more than sequential ones."""
    msg = str(e).lower()
    return "429" in msg or "rate limit" in msg or "resource_exhausted" in msg


def search_one(sub_question: str, max_retries: int = 2) -> Dict:
    """Research a single sub-question. Never raises. Retries specifically on
    rate-limit errors with backoff (new, Day 17) before falling back to a
    failed status — this is the prerequisite Day 15 flagged before shipping
    real parallelism."""
    start = time.time()
    last_error = None

    for attempt in range(max_retries + 1):
        try:
            raw = research_with_fallback(sub_question)
            sources = _dedupe_sources(raw.get("sources", []))
            answer = (raw.get("answer") or "").strip()
            status = "ok" if (answer and sources) else "empty"
            result = {
                "sub_question": sub_question, "answer": answer, "sources": sources,
                "provider": raw.get("provider", "unknown"), "status": status,
                "latency_s": round(time.time() - start, 2),
            }
            log_agent_event(AGENT_NAME, "search_done", status=result["status"],
                            provider=result["provider"], sources=len(result["sources"]),
                            retries=attempt)
            return result
        except Exception as e:
            last_error = e
            if _is_rate_limit_error(e) and attempt < max_retries:
                wait = 2 ** attempt
                log_agent_event(AGENT_NAME, "rate_limited_retrying", sub_question=sub_question,
                                attempt=attempt, wait_s=wait)
                time.sleep(wait)
                continue
            break  # non-rate-limit error, or retries exhausted: fail now

    result = {
        "sub_question": sub_question, "answer": "", "sources": [], "provider": "none",
        "status": "failed", "error": f"{type(last_error).__name__}: {last_error}",
        "latency_s": round(time.time() - start, 2),
    }
    log_agent_event(AGENT_NAME, "search_done", status="failed",
                    provider="none", sources=0)
    return result


def web_search_node(state: Dict) -> Dict:
    """Sequential path — kept for direct testing and as a non-graph entry
    point. The real graph (Day 17) uses search_single_node via Send instead."""
    seen, queries = set(), []
    for q in state.get("plan", []):
        q = q.strip()
        if q and q.lower() not in seen:
            seen.add(q.lower())
            queries.append(q)
    queries = queries[:MAX_QUERIES]

    if not queries:
        return {
            "errors": [{"agent": AGENT_NAME, "error": "empty plan: nothing to search"}],
            "model_usage_log": [{"agent": AGENT_NAME, "status": "skipped_no_input"}],
        }

    results = [search_one(q) for q in queries]
    good = [r for r in results if r["status"] != "failed"]
    failed = [r for r in results if r["status"] == "failed"]

    usage: Dict[str, int] = {}
    for r in good:
        usage[r["provider"]] = usage.get(r["provider"], 0) + 1

    return {
        "search_results": good,
        "errors": [{"agent": AGENT_NAME, "sub_question": r["sub_question"], "error": r["error"]}
                   for r in failed],
        "model_usage_log": [{"agent": AGENT_NAME, "provider": p, "queries": n}
                            for p, n in usage.items()] or [{"agent": AGENT_NAME, "status": "ran_no_results"}],
    }


def search_single_node(state: Dict) -> Dict:
    """Send-target node: handles ONE query, invoked N times in parallel by
    LangGraph's Send mechanism. Each parallel invocation gets an isolated
    state slice ({"query": ...}), and results merge back into the parent
    state via the Annotated[list, add] reducers — verified safe on Day 16."""
    query = state["query"]
    result = search_one(query)

    if result["status"] == "failed":
        return {
            "errors": [{"agent": AGENT_NAME, "sub_question": query, "error": result["error"]}],
            "model_usage_log": [{"agent": AGENT_NAME, "provider": "none", "query": query, "status": "failed"}],
        }

    return {
        "search_results": [result],
        "model_usage_log": [{"agent": AGENT_NAME, "provider": result["provider"], "query": query}],
    }

Overwriting agents/web_search_agent.py


In [3]:
%%writefile agents/supervisor_agent.py
from typing import Dict, List

from langgraph.types import Send
from tools.llm_clients import call_llm
from tools.logger import log_agent_event
from agents.web_search_agent import MAX_QUERIES

AGENT_NAME = "supervisor_agent"
MAX_ITERATIONS = 8

PIPELINE = [
    ("web_search_agent", "web_search"),
    ("data_extraction_agent", "data_extraction"),
    ("fact_checking_agent", "fact_checking"),
    ("summarization_agent", "summarization"),
    ("report_generator_agent", "report_generator"),
]


def _build_plan_prompt(query: str) -> str:
    return f"""Break this research question into 2-5 specific, researchable sub-questions:
"{query}"

Respond with each sub-question on its own line, nothing else. No numbering, no preamble.
"""


def generate_plan(query: str) -> List[str]:
    try:
        raw = call_llm(_build_plan_prompt(query))
        sub_questions = [line.strip("- ").strip() for line in raw.split("\n") if line.strip()]
        return sub_questions[:5] if sub_questions else [query]
    except Exception as e:
        log_agent_event(AGENT_NAME, "plan_generation_failed", error=f"{type(e).__name__}: {e}")
        return [query]


def _agents_completed(state: Dict) -> set:
    return {entry.get("agent") for entry in state.get("model_usage_log", []) if entry.get("agent")}


def supervisor_node(state: Dict) -> Dict:
    iteration = state.get("iteration_count", 0) + 1

    if iteration > MAX_ITERATIONS:
        log_agent_event(AGENT_NAME, "max_iterations_exceeded", iteration=iteration)
        return {"iteration_count": iteration, "next_agent": "failed"}

    updates: Dict = {"iteration_count": iteration}

    if not state.get("plan"):
        plan = generate_plan(state["query"])
        updates["plan"] = plan
        updates["next_agent"] = "web_search"
        log_agent_event(AGENT_NAME, "plan_generated", sub_questions=len(plan))
        return updates

    completed = _agents_completed(state)
    for agent_name, node_name in PIPELINE:
        if agent_name not in completed:
            updates["next_agent"] = node_name
            log_agent_event(AGENT_NAME, "routing", decision=node_name, iteration=iteration)
            return updates

    updates["next_agent"] = "done"
    log_agent_event(AGENT_NAME, "pipeline_complete", iteration=iteration)
    return updates


def route_from_supervisor(state: Dict):
    """Returns either a plain node-name string (unchanged stages) or a list of
    Send objects (NEW, Day 17) to fan out web_search into parallel branches —
    one per sub-question, capped at MAX_QUERIES, deduped."""
    next_agent = state.get("next_agent", "failed")

    if next_agent == "web_search":
        seen, queries = set(), []
        for q in state.get("plan", []):
            q = q.strip()
            if q and q.lower() not in seen:
                seen.add(q.lower())
                queries.append(q)
        queries = queries[:MAX_QUERIES]

        if not queries:
            return "data_extraction"  # nothing to search; let downstream handle empty input

        return [Send("web_search_single", {"query": q}) for q in queries]

    return next_agent

Overwriting agents/supervisor_agent.py


In [4]:
%%writefile graph/build_graph.py
from langgraph.graph import StateGraph, START, END
from state.schema import ResearchState
from agents.supervisor_agent import supervisor_node, route_from_supervisor
from agents.web_search_agent import search_single_node
from agents.data_extraction_agent import data_extraction_node
from agents.fact_checking_agent import fact_checking_node
from agents.summarization_agent import summarization_node
from agents.report_generator_agent import report_generator_node


def build_graph():
    graph = StateGraph(ResearchState)

    graph.add_node("supervisor", supervisor_node)
    graph.add_node("web_search_single", search_single_node)  # Send target — parallel fan-out
    graph.add_node("data_extraction", data_extraction_node)
    graph.add_node("fact_checking", fact_checking_node)
    graph.add_node("summarization", summarization_node)
    graph.add_node("report_generator", report_generator_node)

    graph.add_edge(START, "supervisor")
    graph.add_conditional_edges("supervisor", route_from_supervisor, {
        "web_search_single": "web_search_single",
        "data_extraction": "data_extraction",
        "fact_checking": "fact_checking",
        "summarization": "summarization",
        "report_generator": "report_generator",
        "done": END,
        "failed": END,
    })
    # all N parallel web_search_single branches converge back to supervisor in
    # one superstep (LangGraph's Pregel-style barrier), same as every other stage
    for node in ["web_search_single", "data_extraction", "fact_checking", "summarization", "report_generator"]:
        graph.add_edge(node, "supervisor")

    return graph.compile()

Overwriting graph/build_graph.py


In [5]:
from graph.build_graph import build_graph
import time

app = build_graph()

initial_state = {
    "query": "Compare EV battery strategies of Tesla, BYD, and CATL",
    "plan": [], "next_agent": "", "iteration_count": 0, "needs_more_research": False,
    "search_results": [], "extracted_facts": [], "fact_check_results": [], "summaries": [],
    "errors": [], "final_report": None, "model_usage_log": [],
}

start = time.time()
result = app.invoke(initial_state)
elapsed = time.time() - start

print(f"Total time: {elapsed:.2f}s | Iterations: {result['iteration_count']}")
print(f"Plan ({len(result['plan'])} sub-questions):")
for q in result["plan"]:
    print(" -", q)
print(f"\nSearch results: {len(result['search_results'])} | Errors: {len(result['errors'])}")
for r in result["search_results"]:
    print(f"  [{r['status']}] {r['sub_question']}")
print(f"\nReport length: {len(result['final_report'])} chars")

2026-10-09 16:44:40,314 | INFO | research_system | [supervisor_agent] plan_generated sub_questions=4
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


 Gemini unavailable (ClientError), falling back to Tavily...
 Gemini unavailable (ClientError), falling back to Tavily...
 Gemini unavailable (ClientError), falling back to Tavily...
 Gemini unavailable (ClientError), falling back to Tavily...


2026-10-09 16:44:43,971 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-09 16:44:44,066 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-09 16:44:44,216 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-09 16:44:44,224 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-09 16:44:44,227 | INFO | research_system | [supervisor_agent] routing decision=data_extraction iteration=2
2026-10-09 16:44:44,779 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=How do Tesla, BYD, and CATL differ in their core battery cell chemistries and manufacturing processes? facts_found=10
2026-10-09 16:44:45,027 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the distinct approaches of these three

Total time: 43.80s | Iterations: 6
Plan (4 sub-questions):
 - How do Tesla, BYD, and CATL differ in their core battery cell chemistries and manufacturing processes?
 - What are the distinct approaches of these three companies regarding vertical integration versus supply chain partnerships?
 - How do the energy density, cost per kilowatt-hour, and degradation rates compare across the battery products of Tesla, BYD, and CATL?
 - What are the primary strategic advantages and vulnerabilities of each company's battery technology roadmap in the global EV market?

Search results: 4 | Errors: 0
  [ok] How do Tesla, BYD, and CATL differ in their core battery cell chemistries and manufacturing processes?
  [ok] What are the distinct approaches of these three companies regarding vertical integration versus supply chain partnerships?
  [ok] How do the energy density, cost per kilowatt-hour, and degradation rates compare across the battery products of Tesla, BYD, and CATL?
  [ok] What are the prima

In [6]:
%%writefile tests/test_graph_integration.py
"""Integration tests for the full graph, including Day 17's Send-based fan-out."""
import agents.supervisor_agent as sup
from graph.build_graph import build_graph


def _mock_node(name, **extra_state):
    def node(state):
        return {"model_usage_log": [{"agent": name}], **extra_state}
    return node


def test_full_pipeline_runs_in_correct_order(monkeypatch):
    call_order = []

    def tracking_node(name, **extra_state):
        def node(state):
            call_order.append(name)
            return {"model_usage_log": [{"agent": name}], **extra_state}
        return node

    monkeypatch.setattr(sup, "generate_plan", lambda q: ["sub-q 1"])

    import graph.build_graph as gb
    monkeypatch.setattr(gb, "search_single_node", tracking_node("web_search_agent", search_results=[{"x": 1}]))
    monkeypatch.setattr(gb, "data_extraction_node", tracking_node("data_extraction_agent", extracted_facts=[{"x": 1}]))
    monkeypatch.setattr(gb, "fact_checking_node", tracking_node("fact_checking_agent", fact_check_results=[{"x": 1}]))
    monkeypatch.setattr(gb, "summarization_node", tracking_node("summarization_agent", summaries=[{"x": 1}]))
    monkeypatch.setattr(gb, "report_generator_node", tracking_node("report_generator_agent", final_report="done"))

    app = gb.build_graph()
    initial_state = {
        "query": "test query", "plan": [], "next_agent": "", "iteration_count": 0,
        "needs_more_research": False, "search_results": [], "extracted_facts": [],
        "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
        "model_usage_log": [],
    }
    result = app.invoke(initial_state)

    assert call_order == [
        "web_search_agent", "data_extraction_agent", "fact_checking_agent",
        "summarization_agent", "report_generator_agent",
    ]
    assert result["final_report"] == "done"


def test_web_search_fans_out_to_multiple_parallel_sends(monkeypatch):
    """The new Day 17 behavior: one Send per sub-question, all running before
    the graph proceeds to data_extraction."""
    call_count = {"value": 0}

    def counting_search(state):
        call_count["value"] += 1
        return {"search_results": [{"sub_question": state["query"], "status": "ok"}],
                "model_usage_log": [{"agent": "web_search_agent"}]}

    monkeypatch.setattr(sup, "generate_plan", lambda q: ["q1", "q2", "q3"])

    import graph.build_graph as gb
    monkeypatch.setattr(gb, "search_single_node", counting_search)
    monkeypatch.setattr(gb, "data_extraction_node", _mock_node("data_extraction_agent", extracted_facts=[]))
    monkeypatch.setattr(gb, "fact_checking_node", _mock_node("fact_checking_agent", fact_check_results=[]))
    monkeypatch.setattr(gb, "summarization_node", _mock_node("summarization_agent", summaries=[]))
    monkeypatch.setattr(gb, "report_generator_node", _mock_node("report_generator_agent", final_report="ok"))

    app = gb.build_graph()
    initial_state = {
        "query": "test query", "plan": [], "next_agent": "", "iteration_count": 0,
        "needs_more_research": False, "search_results": [], "extracted_facts": [],
        "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
        "model_usage_log": [],
    }
    result = app.invoke(initial_state)

    assert call_count["value"] == 3  # one Send per sub-question
    assert len(result["search_results"]) == 3


def test_graph_terminates_on_empty_plan_fallback(monkeypatch):
    monkeypatch.setattr(sup, "call_llm", lambda p: (_ for _ in ()).throw(RuntimeError("down")))

    import graph.build_graph as gb
    monkeypatch.setattr(gb, "search_single_node", lambda state: {
        "search_results": [{"sub_question": state["query"]}],
        "model_usage_log": [{"agent": "web_search_agent"}]})
    monkeypatch.setattr(gb, "data_extraction_node", _mock_node("data_extraction_agent", extracted_facts=[]))
    monkeypatch.setattr(gb, "fact_checking_node", _mock_node("fact_checking_agent", fact_check_results=[]))
    monkeypatch.setattr(gb, "summarization_node", _mock_node("summarization_agent", summaries=[]))
    monkeypatch.setattr(gb, "report_generator_node", _mock_node("report_generator_agent", final_report="ok"))

    app = gb.build_graph()
    initial_state = {
        "query": "test query", "plan": [], "next_agent": "", "iteration_count": 0,
        "needs_more_research": False, "search_results": [], "extracted_facts": [],
        "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
        "model_usage_log": [],
    }
    result = app.invoke(initial_state)
    assert result["plan"] == ["test query"]
    assert result["final_report"] == "ok"


def test_graph_respects_max_iterations(monkeypatch):
    import graph.build_graph as gb
    broken_node = lambda state: {}

    monkeypatch.setattr(sup, "generate_plan", lambda q: ["sub-q 1"])
    monkeypatch.setattr(gb, "search_single_node", broken_node)
    monkeypatch.setattr(gb, "data_extraction_node", broken_node)
    monkeypatch.setattr(gb, "fact_checking_node", broken_node)
    monkeypatch.setattr(gb, "summarization_node", broken_node)
    monkeypatch.setattr(gb, "report_generator_node", broken_node)

    app = gb.build_graph()
    initial_state = {
        "query": "test query", "plan": [], "next_agent": "", "iteration_count": 0,
        "needs_more_research": False, "search_results": [], "extracted_facts": [],
        "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
        "model_usage_log": [],
    }
    result = app.invoke(initial_state)
    assert result["iteration_count"] <= sup.MAX_ITERATIONS + 1
    assert result["final_report"] is None

Overwriting tests/test_graph_integration.py


In [ ]:
%%writefile tests/test_web_search_parallel.py
import agents.web_search_agent as wsa


def test_rate_limit_detection():
    assert wsa._is_rate_limit_error(Exception("429 Too Many Requests"))
    assert wsa._is_rate_limit_error(Exception("RESOURCE_EXHAUSTED: quota"))
    assert wsa._is_rate_limit_error(Exception("rate limit reached"))
    assert not wsa._is_rate_limit_error(Exception("connection reset"))


def test_search_one_retries_on_rate_limit_then_succeeds(monkeypatch):
    calls = {"count": 0}
    def flaky(q):
        calls["count"] += 1
        if calls["count"] < 2:
            raise Exception("429 rate limit reached")
        return {"answer": "ok", "sources": [{"url": "http://a"}], "provider": "tavily"}
    monkeypatch.setattr(wsa, "research_with_fallback", flaky)
    monkeypatch.setattr(wsa.time, "sleep", lambda s: None)  # skip real waiting in tests

    result = wsa.search_one("q", max_retries=2)
    assert result["status"] == "ok"
    assert calls["count"] == 2


def test_search_one_does_not_retry_non_rate_limit_errors(monkeypatch):
    calls = {"count": 0}
    def boom(q):
        calls["count"] += 1
        raise Exception("connection reset")
    monkeypatch.setattr(wsa, "research_with_fallback", boom)

    result = wsa.search_one("q", max_retries=2)
    assert result["status"] == "failed"
    assert calls["count"] == 1  # no retry for non-rate-limit errors


def test_search_single_node_success(monkeypatch):
    monkeypatch.setattr(wsa, "research_with_fallback",
                        lambda q: {"answer": "x", "sources": [{"url": "http://a"}], "provider": "tavily"})
    out = wsa.search_single_node({"query": "test question"})
    assert len(out["search_results"]) == 1
    assert out["search_results"][0]["sub_question"] == "test question"
    assert out["model_usage_log"][0]["agent"] == "web_search_agent"


def test_search_single_node_failure_goes_to_errors(monkeypatch):
    def boom(q):
        raise Exception("connection reset")
    monkeypatch.setattr(wsa, "research_with_fallback", boom)
    out = wsa.search_single_node({"query": "test question"})
    assert "errors" in out
    assert "search_results" not in out